# Simulação do Protocolo de Árvore Geradora (STP / RSTP) no Sistema SCADA-Core
## Estudo de Caso: Complexo Industrial de Fertilizantes Químicos NPK (ECAA08)

Este notebook modela a topologia Ethernet industrial (Camada 2) que interliga os switches gerenciáveis dos setores da **Planta Química NPK** (`SCADA-Core`, Setor 100 - Reator `R-101`, Setor 200 - Granulação/Secagem, Setor 300 - Parque de Tancagem e Setor 400 - Logística/AGV) como um grafo ponderado $G = (V, E, w)$.

### Objetivos da Simulação:
1. **Eleição Determinística do *Root Bridge*** com base na menor tupla $\text{BID} = (\text{Bridge Priority}, \text{MAC Address})$.
2. **Cálculo de *Root Path Cost*** utilizando os pesos padrão da norma **IEEE 802.1w (RSTP)**.
3. **Atribuição de Papéis e Estados de Porta** (*Root Port*, *Designated Port*, *Alternate Port / Discarding*) para prevenção de ciclos (*Broadcast Storms*).
4. **Simulação de Falha Física (*Link Down*)** no backbone principal do Reator `R-101` e verificação da reconvergência automática.
5. **Auditoria Formal em Matemática Discreta** comprovando que o subgrafo resultante $T = (V, E_T)$ é uma Árvore Geradora ($|E_T| = |V| - 1$, conexa e acíclica).

### 1. Importação de Bibliotecas e Modelagem do Grafo da Planta NPK
Definimos os switches industriais de cada setor com suas prioridades configuradas manualmente para garantir determinismo operacional (`SW-CORE` como *Root Bridge* primário com prioridade `4096` e `SW-S100` como *Root Bridge* secundário com prioridade `8192`).

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
import pandas as pd

# ==============================================================================
# 1. DEFINIÇÃO DOS VÉRTICES (SWITCHES INDUSTRIAIS DA PLANTA NPK)
# ==============================================================================
# Bridge ID (BID) = (Bridge_Priority, MAC_Address) -> Menor BID vence a eleição
switches = {
    "SW-CORE": {"setor": "Sala de Controle (SCADA-Core)", "priority": 4096,  "mac": "00:1A:2B:00:00:01"},
    "SW-S100": {"setor": "Setor 100 (Reator R-101)",      "priority": 8192,  "mac": "00:1A:2B:00:00:02"},
    "SW-S300": {"setor": "Setor 300 (Tancagem NH3/H3PO4)","priority": 32768, "mac": "00:1A:2B:00:00:03"},
    "SW-S200": {"setor": "Setor 200 (Granulação/Secagem)","priority": 32768, "mac": "00:1A:2B:00:00:04"},
    "SW-S400": {"setor": "Setor 400 (Logística/AGV-401)", "priority": 32768, "mac": "00:1A:2B:00:00:05"},
}

# ==============================================================================
# 2. DEFINIÇÃO DAS ARESTAS (ENLACES FÍSICOS E CUSTOS IEEE 802.1w)
# ==============================================================================
# Custos padrão RSTP: 10 Gbps = 2.000 | 1 Gbps = 20.000 | 100 Mbps = 200.000
enlaces = [
    ("SW-CORE", "SW-S100", {"cost": 2000,  "bandwidth": "10 Gbps (Fibra Backbone)"}),
    ("SW-CORE", "SW-S400", {"cost": 20000, "bandwidth": "1 Gbps (Fibra Anel)"}),
    ("SW-S100", "SW-S300", {"cost": 20000, "bandwidth": "1 Gbps (Fibra Anel)"}),
    ("SW-S300", "SW-S200", {"cost": 20000, "bandwidth": "1 Gbps (Fibra Anel)"}),
    ("SW-S200", "SW-S400", {"cost": 20000, "bandwidth": "1 Gbps (Fibra Anel)"}),
    # Enlace redundante cruzado entre Reator R-101 e Granulação (malha parcial)
    ("SW-S100", "SW-S200", {"cost": 200000,"bandwidth": "100 Mbps (Cabo Cobre Backup)"}),
]

# Construção do Grafo Não-Direcionado G = (V, E, w)
G = nx.Graph()
for sw, attr in switches.items():
    G.add_node(sw, **attr)
G.add_edges_from(enlaces)

print(f"Grafo da Rede SCADA criado com |V| = {G.number_of_nodes()} nós e |E| = {G.number_of_edges()} enlaces físicos.")
print(f"Ciclos independentes detectados (número ciclomático |E| - |V| + 1): {G.number_of_edges() - G.number_of_nodes() + 1}")

### 2. Implementação do Algoritmo RSTP (Eleição de Root, Árvore Geradora e Papéis de Porta)
A função `executar_rstp(grafo)` reproduz o comportamento convergente do protocolo **IEEE 802.1w**:
1. Identifica o *Root Bridge* com menor **Bridge ID**.
2. Calcula a árvore de caminhos mínimos (*Shortest-Path Tree*) a partir do *Root Bridge*, utilizando o menor BID do nó vizinho como critério determinístico de desempate.
3. Classifica cada extremidade das arestas em:
   - **Root Port (RP):** Porta de menor custo em direção ao *Root Bridge* (estado *Forwarding*).
   - **Designated Port (DP):** Porta responsável por encaminhar quadros no segmento de rede (estado *Forwarding*).
   - **Alternate Port (AP):** Porta redundante que fecha um ciclo, mantida logicamente bloqueada (estado *Discarding*).

In [ ]:
def executar_rstp(grafo):
    """
    Simula a convergência do protocolo RSTP (IEEE 802.1w) sobre um grafo L2.
    Retorna o Root Bridge, a Árvore Geradora ativa (T) e a tabela de estados das portas.
    """
    if not nx.is_connected(grafo):
        raise ValueError("ALERTA CRÍTICO: O grafo da rede está particionado (desconexo)!")

    # Passo 1: Eleição do Root Bridge (menor tupla (priority, mac))
    root_bridge = min(
        grafo.nodes,
        key=lambda n: (grafo.nodes[n]["priority"], grafo.nodes[n]["mac"])
    )

    # Passo 2: Cálculo do Root Path Cost mínimo e escolha da Root Port (RP)
    root_path_cost = {n: float("inf") for n in grafo.nodes}
    root_port_neighbor = {n: None for n in grafo.nodes}
    root_path_cost[root_bridge] = 0

    visitados = set()
    while len(visitados) < len(grafo.nodes):
        u = min(
            (n for n in grafo.nodes if n not in visitados),
            key=lambda n: (root_path_cost[n], grafo.nodes[n]["priority"], grafo.nodes[n]["mac"])
        )
        visitados.add(u)

        for v in grafo.neighbors(u):
            if v in visitados:
                continue
            custo_enlace = grafo.edges[u, v]["cost"]
            novo_custo = root_path_cost[u] + custo_enlace

            atual_pai = root_port_neighbor[v]
            bid_u = (grafo.nodes[u]["priority"], grafo.nodes[u]["mac"])
            bid_atual = (grafo.nodes[atual_pai]["priority"], grafo.nodes[atual_pai]["mac"]) if atual_pai else (float("inf"), "")

            if (novo_custo < root_path_cost[v]) or (novo_custo == root_path_cost[v] and bid_u < bid_atual):
                root_path_cost[v] = novo_custo
                root_port_neighbor[v] = u

    # Passo 3: Classificação dos Papéis e Estados de Cada Porta nos Enlaces
    arestas_ativas = []
    arestas_bloqueadas = []
    tabela_portas = []

    for u, v, data in grafo.edges(data=True):
        if root_port_neighbor[v] == u:
            papel_u, estado_u = "Designated Port (DP)", "Forwarding"
            papel_v, estado_v = "Root Port (RP)", "Forwarding"
            arestas_ativas.append((u, v))
        elif root_port_neighbor[u] == v:
            papel_u, estado_u = "Root Port (RP)", "Forwarding"
            papel_v, estado_v = "Designated Port (DP)", "Forwarding"
            arestas_ativas.append((u, v))
        else:
            criterio_u = (root_path_cost[u], grafo.nodes[u]["priority"], grafo.nodes[u]["mac"])
            criterio_v = (root_path_cost[v], grafo.nodes[v]["priority"], grafo.nodes[v]["mac"])

            if criterio_u < criterio_v:
                papel_u, estado_u = "Designated Port (DP)", "Forwarding"
                papel_v, estado_v = "Alternate Port (AP)", "Discarding (Bloqueada)"
            else:
                papel_u, estado_u = "Alternate Port (AP)", "Discarding (Bloqueada)"
                papel_v, estado_v = "Designated Port (DP)", "Forwarding"
            arestas_bloqueadas.append((u, v))

        tabela_portas.append({
            "Enlace": f"{u} <--> {v}",
            "Custo": data["cost"],
            f"Papel em {u}": f"{papel_u} [{estado_u}]",
            f"Papel em {v}": f"{papel_v} [{estado_v}]",
            "Status Enlace": "ATIVO (Árvore T)" if (u, v) in arestas_ativas else "BLOQUEADO (Loop Prevention)"
        })

    return root_bridge, root_path_cost, arestas_ativas, arestas_bloqueadas, pd.DataFrame(tabela_portas)

root, custos, ativas, bloqueadas, df_rstp = executar_rstp(G)

print("=== RESULTADO DA CONVERGÊNCIA RSTP (OPERAÇÃO NORMAL) ===")
print(f"Root Bridge Eleito: {root} (Priority: {G.nodes[root]['priority']}, MAC: {G.nodes[root]['mac']})\n")
print("Custo de Caminho até o Root Bridge por Setor:")
for sw, c in custos.items():
    print(f"  - {sw:8s} ({G.nodes[sw]['setor']:30s}): Custo = {c}")

print("\nTabela de Estados das Portas:")
df_rstp

### 3. Visualização Gráfica da Topologia Física vs. Árvore Geradora Lógica
Representação visual do grafo da rede SCADA-Core:
- **Nó Dourado:** *Root Bridge* (`SW-CORE`).
- **Arestas Verdes Contínuas:** Enlaces ativos na Árvore Geradora $T$ (*Forwarding*).
- **Arestas Vermelhas Tracejadas:** Enlaces bloqueados logicamente (*Alternate Port / Discarding*) para eliminar ciclos.

In [ ]:
def plotar_topologia_rstp(grafo, root, ativas, bloqueadas, titulo_extra="Operação Normal"):
    pos = {
        "SW-CORE": (0.5, 1.0),
        "SW-S100": (0.1, 0.6),
        "SW-S400": (0.9, 0.6),
        "SW-S300": (0.2, 0.1),
        "SW-S200": (0.8, 0.1),
    }

    fig, ax = plt.subplots(figsize=(11, 6))

    # Cores dos nós (Ouro para Root Bridge, Azul Industrial para os demais)
    node_colors = ["#FFD700" if n == root else "#4A90E2" for n in grafo.nodes]

    # Desenhar nós
    nx.draw_networkx_nodes(grafo, pos, node_color=node_colors, node_size=2600, edgecolors="black", linewidths=2, ax=ax)
    nx.draw_networkx_labels(grafo, pos, font_size=10, font_weight="bold", ax=ax)

    # Desenhar arestas ativas (Árvore Geradora T)
    nx.draw_networkx_edges(grafo, pos, edgelist=ativas, width=3.5, edge_color="#2ECC71", ax=ax, label="Enlace Forwarding (Árvore T)")

    # Desenhar arestas bloqueadas (Alternate Port / Discarding)
    nx.draw_networkx_edges(grafo, pos, edgelist=bloqueadas, width=2.5, edge_color="#E74C3C", style="dashed", ax=ax, label="Enlace Discarding (Bloqueado)")

    # Rótulos de custo nos enlaces
    edge_labels = {(u, v): f"w = {d['cost']}" for u, v, d in grafo.edges(data=True)}
    nx.draw_networkx_edge_labels(grafo, pos, edge_labels=edge_labels, font_size=9, font_color="#2C3E50", ax=ax)

    ax.set_title(f"Topologia Lógica RSTP — Complexo NPK ({titulo_extra})\nRoot Bridge: {root}", fontsize=13, fontweight="bold")
    ax.legend(loc="lower center", bbox_to_anchor=(0.5, -0.08), ncol=2, frameon=True)
    ax.axis("off")
    plt.tight_layout()
    plt.show()

plotar_topologia_rstp(G, root, ativas, bloqueadas, "Operação Normal")

### 4. Simulação de Falha Crítica (Rompimento do Backbone `SW-CORE` $\leftrightarrow$ `SW-S100`)
Simulamos o rompimento físico da fibra óptica de $10\text{ Gbps}$ que conecta a Sala de Controle (`SW-CORE`) ao setor de maior risco químico (`SW-S100` — Reator *Pre-Neutralizer* `R-101`). Observa-se a transição instantânea da porta *Alternate* entre `SW-S300` e `SW-S200` para o estado *Forwarding*, restabelecendo a telemetria de `PT-101`, `TT-101` e `AT-101`.

In [ ]:
# Criando cópia do grafo para simular falha física no enlace principal do Reator R-101
G_falha = G.copy()
enlace_rompido = ("SW-CORE", "SW-S100")
G_falha.remove_edge(*enlace_rompido)

print(f"⚠️ SIMULAÇÃO DE FALHA: Rompimento físico do enlace {enlace_rompido[0]} <--> {enlace_rompido[1]}!")

root_f, custos_f, ativas_f, bloqueadas_f, df_rstp_f = executar_rstp(G_falha)

print("\nNovos Custos de Caminho após Reconvergência RSTP (< 1 segundo):")
for sw, c in custos_f.items():
    print(f"  - {sw:8s}: Novo Custo = {c} (Anterior: {custos[sw]})")

plotar_topologia_rstp(G_falha, root_f, ativas_f, bloqueadas_f, "Após Falha no Backbone SW-CORE <-> SW-S100")
df_rstp_f

### 5. Validação Matemática de Propriedades de Árvore (Teoria dos Grafos)
Verificação formal de que o subgrafo lógico resultante $T = (V, E_{\text{ativas}})$ satisfaz as condições necessárias e suficientes de uma **Árvore Geradora**:
1. $V_T = V_G$ (todos os switches dos setores permanecem integrados ao domínio de supervisão).
2. $|E_T| = |V_T| - 1$ (ausência de arestas redundantes ativas).
3. $T$ é **conexo** e **acíclico**.

In [ ]:
def auditar_arvore_geradora(grafo_original, arestas_ativas):
    T = nx.Graph()
    T.add_nodes_from(grafo_original.nodes(data=True))
    T.add_edges_from(arestas_ativas)

    n_vertices = T.number_of_nodes()
    n_arestas = T.number_of_edges()
    conexo = nx.is_connected(T)
    aciclico = nx.is_forest(T) and conexo
    e_arvore = nx.is_tree(T)

    print("=== AUDITORIA MATEMÁTICA DO SUBGRAFO LÓGICO T ===")
    print(f"1. Vértices preservados (|V_T| == |V_G|): {n_vertices == grafo_original.number_of_nodes()} ({n_vertices} nós)")
    print(f"2. Propriedade |E_T| = |V_T| - 1:          {n_arestas == n_vertices - 1} ({n_arestas} arestas ativas)")
    print(f"3. Grafo Conexo (Todos setores alcançáveis): {conexo}")
    print(f"4. Grafo Acíclico (Zero Broadcast Storms):   {aciclico}")
    print(f"5. Resultado Formal (T é Árvore Geradora?):  {e_arvore}")

auditar_arvore_geradora(G_falha, ativas_f)